In [1]:
# Install audio deps (Kaggle images usually have torch; librosa/soundfile may need install)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "librosa", "soundfile", "audioread", "seaborn", "tqdm"])

0

In [2]:
# =============================================================================
# CUDA / GPU SETUP
# =============================================================================
import os
import inspect
import random
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import librosa
import soundfile as sf

from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_score, recall_score, f1_score,
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# CUDA optimizations
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.enabled = True

print("=" * 60)
print("DEVICE CHECK")
print("=" * 60)
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version    : {torch.version.cuda}")
    print(f"GPU count       : {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        props = torch.cuda.get_device_properties(i)
        print(f"         Memory: {props.total_memory / 1e9:.1f} GB")
print(f"Using device    : {DEVICE}")
print("=" * 60)

# Quick GPU smoke test
if torch.cuda.is_available():
    _x = torch.randn(4, 4, device=DEVICE)
    _y = _x @ _x.T
    print(f"GPU smoke test OK — tensor on {_y.device}")
else:
    print("WARNING: No GPU detected. Enable GPU in Kaggle settings for faster training.")

DEVICE CHECK
PyTorch version : 2.10.0+cu128
CUDA available  : True
CUDA version    : 12.8
GPU count       : 2
  GPU 0: Tesla T4
         Memory: 15.6 GB
  GPU 1: Tesla T4
         Memory: 15.6 GB
Using device    : cuda
GPU smoke test OK — tensor on cuda:0


In [3]:
# =============================================================================
# CONFIGURATION (Kaggle-aware paths)
# =============================================================================
IS_KAGGLE = Path("/kaggle/input").exists()

def _find_kaggle_data_root():
    candidates = [
        Path("/kaggle/input/datasets/siddhanta98/cadence-dataset"),
        Path("/kaggle/input/cadence-dataset"),
    ]
    # Also scan /kaggle/input for any folder containing splits/train
    kaggle_in = Path("/kaggle/input")
    if kaggle_in.exists():
        for p in sorted(kaggle_in.rglob("splits/train")):
            candidates.insert(0, p.parent.parent)
    for root in candidates:
        if (root / "splits" / "train").exists():
            return root
    return candidates[0]

if IS_KAGGLE:
    WORK_DIR = Path("/kaggle/working")
    DATA_ROOT = _find_kaggle_data_root()
    LEGACY_SPLITS_DIR = DATA_ROOT / "splits"
else:
    # Local fallback (same layout as project)
    WORK_DIR = Path(".").resolve()
    DATA_ROOT = WORK_DIR / "dataset"
    LEGACY_SPLITS_DIR = DATA_ROOT / "splits"

PROCESSED_CQT_DIR = WORK_DIR / "processed_cqt"
MODELS_DIR = WORK_DIR / "models"
RESULTS_DIR = WORK_DIR / "results"

for d in [PROCESSED_CQT_DIR / "train", PROCESSED_CQT_DIR / "test", MODELS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Project settings
PROJECT_NAME = "Homie Minister SOTA"
GENRES = ["tamang_selo", "deuda", "bhajan", "newari", "tharu", "lok_dohori"]
SAMPLE_RATE = 22050
DURATION = 30
N_BINS = 84
BINS_PER_OCTAVE = 12
HOP_LENGTH = 512
TARGET_FRAMES = 1292

# Training hyperparameters (tuned for Kaggle GPU session)
BATCH_SIZE = 16
NUM_EPOCHS = 25
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2 if IS_KAGGLE and torch.cuda.is_available() else 0

def print_settings():
    print(f"=== {PROJECT_NAME} ===")
    print(f"Environment     : {'Kaggle' if IS_KAGGLE else 'Local'}")
    print(f"Data root       : {DATA_ROOT}")
    print(f"Splits dir      : {LEGACY_SPLITS_DIR}")
    print(f"Processed CQT   : {PROCESSED_CQT_DIR}")
    print(f"Models dir      : {MODELS_DIR}")
    print(f"Genres          : {GENRES}")
    print(f"Sample rate     : {SAMPLE_RATE} Hz")
    print(f"CQT bins        : {N_BINS} (bins/octave: {BINS_PER_OCTAVE})")
    print(f"Batch / Epochs  : {BATCH_SIZE} / {NUM_EPOCHS}")
    print("=" * 25)

print_settings()

# Verify dataset exists
for split in ["train", "test"]:
    split_dir = LEGACY_SPLITS_DIR / split
    if not split_dir.exists():
        raise FileNotFoundError(f"Missing split directory: {split_dir}")
    counts = {}
    for genre in GENRES:
        gdir = split_dir / genre
        counts[genre] = len(list(gdir.glob("*.wav"))) if gdir.exists() else 0
    print(f"{split}: {sum(counts.values())} wav files — {counts}")

=== Homie Minister SOTA ===
Environment     : Kaggle
Data root       : /kaggle/input/datasets/siddhanta98/cadence-dataset
Splits dir      : /kaggle/input/datasets/siddhanta98/cadence-dataset/splits
Processed CQT   : /kaggle/working/processed_cqt
Models dir      : /kaggle/working/models
Genres          : ['tamang_selo', 'deuda', 'bhajan', 'newari', 'tharu', 'lok_dohori']
Sample rate     : 22050 Hz
CQT bins        : 84 (bins/octave: 12)
Batch / Epochs  : 16 / 25
train: 2903 wav files — {'tamang_selo': 533, 'deuda': 528, 'bhajan': 413, 'newari': 420, 'tharu': 535, 'lok_dohori': 474}
test: 810 wav files — {'tamang_selo': 154, 'deuda': 141, 'bhajan': 166, 'newari': 115, 'tharu': 110, 'lok_dohori': 124}


In [4]:
# =============================================================================
# MODEL — FolkMusicCQTResNet (2-channel CQT ResNet)
# =============================================================================
class ResBlock2D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout_prob=0.15):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.dropout = nn.Dropout2d(p=dropout_prob)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)


class FolkMusicCQTResNet(nn.Module):
    def __init__(self, num_classes=6, dropout_prob=0.2):
        super().__init__()
        self.conv_init = nn.Conv2d(2, 32, 3, padding=1, bias=False)
        self.bn_init = nn.BatchNorm2d(32)
        self.layer1 = ResBlock2D(32, 64, dropout_prob=dropout_prob)
        self.pool1 = nn.MaxPool2d(2)
        self.layer2 = ResBlock2D(64, 128, dropout_prob=dropout_prob)
        self.pool2 = nn.MaxPool2d(2)
        self.layer3 = ResBlock2D(128, 256, dropout_prob=dropout_prob)
        self.pool3 = nn.MaxPool2d((1, 2))
        self.layer4 = ResBlock2D(256, 256, dropout_prob=dropout_prob)
        self.pool4 = nn.MaxPool2d(2)
        self.layer5 = ResBlock2D(256, 512, dropout_prob=dropout_prob)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Sequential(
            nn.Linear(512, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Dropout(p=dropout_prob), nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = F.relu(self.bn_init(self.conv_init(x)))
        x = self.pool1(self.layer1(x))
        x = self.pool2(self.layer2(x))
        x = self.pool3(self.layer3(x))
        x = self.pool4(self.layer4(x))
        x = self.layer5(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

# Sanity check
_model = FolkMusicCQTResNet(num_classes=len(GENRES)).to(DEVICE)
_dummy = torch.randn(2, 2, N_BINS, TARGET_FRAMES, device=DEVICE)
_out = _model(_dummy)
print(f"Model OK — params: {sum(p.numel() for p in _model.parameters()):,}, output: {_out.shape}")
del _model, _dummy, _out
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Model OK — params: 6,128,006, output: torch.Size([2, 6])


In [5]:
# =============================================================================
# PREPROCESSING — HPSS + CQT feature extraction
# =============================================================================

def compute_sota_features(audio_path_or_array, sr=SAMPLE_RATE):
    """Return normalized 2-channel CQT features: (2, N_BINS, N_FRAMES)."""
    if isinstance(audio_path_or_array, (str, Path)):
        audio, _ = librosa.load(str(audio_path_or_array), sr=sr, mono=True)
    else:
        audio = audio_path_or_array

    harmonic, percussive = librosa.effects.hpss(audio)

    try:
        cqt_h = np.abs(librosa.cqt(harmonic, sr=sr, hop_length=HOP_LENGTH,
                                    n_bins=N_BINS, bins_per_octave=BINS_PER_OCTAVE))
        cqt_p = np.abs(librosa.cqt(percussive, sr=sr, hop_length=HOP_LENGTH,
                                    n_bins=N_BINS, bins_per_octave=BINS_PER_OCTAVE))
    except Exception:
        mel_h = librosa.feature.melspectrogram(y=harmonic, sr=sr, n_mels=N_BINS, hop_length=HOP_LENGTH)
        mel_p = librosa.feature.melspectrogram(y=percussive, sr=sr, n_mels=N_BINS, hop_length=HOP_LENGTH)
        cqt_h, cqt_p = np.abs(mel_h), np.abs(mel_p)

    cqt_h_db = librosa.amplitude_to_db(cqt_h, ref=np.max)
    cqt_p_db = librosa.amplitude_to_db(cqt_p, ref=np.max)

    def normalize(db_array):
        amin, amax = db_array.min(), db_array.max()
        denom = amax - amin
        return np.zeros_like(db_array) if denom < 1e-9 else (db_array - amin) / denom

    return np.stack([normalize(cqt_h_db), normalize(cqt_p_db)], axis=0).astype(np.float32)


def pad_or_truncate(features, target_frames=TARGET_FRAMES):
    current = features.shape[2]
    if current > target_frames:
        return features[:, :, :target_frames]
    if current < target_frames:
        pad = target_frames - current
        return np.pad(features, ((0, 0), (0, 0), (0, pad)), mode="constant")
    return features


def preprocess_all_dataset_splits(force=False):
    """Precompute .npy CQT features from wav splits into WORK_DIR/processed_cqt."""
    print("Starting HPSS + CQT precomputation...")
    total_new, total_skip, total_err = 0, 0, 0

    for split in ["train", "test"]:
        input_split = LEGACY_SPLITS_DIR / split
        output_split = PROCESSED_CQT_DIR / split
        if not input_split.exists():
            print(f"Skipping missing split: {input_split}")
            continue

        print(f"\nProcessing '{split}' split...")
        for genre in GENRES:
            in_dir = input_split / genre
            out_dir = output_split / genre
            if not in_dir.exists():
                continue
            out_dir.mkdir(parents=True, exist_ok=True)
            wav_files = sorted(in_dir.glob("*.wav"))
            print(f"  {genre}: {len(wav_files)} files")

            for wav_path in tqdm(wav_files, desc=f"    {genre}", leave=False):
                out_path = out_dir / (wav_path.stem + ".npy")
                if out_path.exists() and not force:
                    total_skip += 1
                    continue
                try:
                    feats = pad_or_truncate(compute_sota_features(wav_path))
                    np.save(str(out_path), feats)
                    total_new += 1
                except Exception as exc:
                    total_err += 1
                    print(f"    Error {wav_path.name}: {exc}")

    print(f"\nPrecomputation done — new: {total_new}, skipped: {total_skip}, errors: {total_err}")
    print(f"Stored in: {PROCESSED_CQT_DIR}")

In [6]:
# =============================================================================
# DATASET & DATALOADERS
# =============================================================================
class FolkMusicCQTDataset(Dataset):
    def __init__(self, split="train", use_precomputed=True):
        self.split = split
        self.use_precomputed = use_precomputed
        self.filepaths, self.labels, self.is_precomputed_list = [], [], []

        processed_split = PROCESSED_CQT_DIR / split
        legacy_split = LEGACY_SPLITS_DIR / split

        for label, genre in enumerate(GENRES):
            proc_genre = processed_split / genre
            legacy_genre = legacy_split / genre

            if self.use_precomputed and proc_genre.exists():
                npy_files = sorted(proc_genre.glob("*.npy"))
                if npy_files:
                    for f in npy_files:
                        self.filepaths.append(f)
                        self.labels.append(label)
                        self.is_precomputed_list.append(True)
                    continue

            if legacy_genre.exists():
                for f in sorted(legacy_genre.glob("*.wav")):
                    self.filepaths.append(f)
                    self.labels.append(label)
                    self.is_precomputed_list.append(False)

        if not self.filepaths:
            raise FileNotFoundError(
                f"No data in '{processed_split}' or '{legacy_split}'. Run preprocessing first."
            )

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        filepath = self.filepaths[idx]
        label = self.labels[idx]
        try:
            if self.is_precomputed_list[idx]:
                features = np.load(str(filepath))
            else:
                features = compute_sota_features(filepath)
        except Exception as exc:
            print(f"[Warning] {filepath}: {exc} — using zero fallback")
            features = np.zeros((2, N_BINS, TARGET_FRAMES), dtype=np.float32)

        features = pad_or_truncate(features)
        return torch.tensor(features, dtype=torch.float32), label


def create_sota_dataloaders(batch_size=BATCH_SIZE, use_precomputed=True, num_workers=NUM_WORKERS):
    pin = torch.cuda.is_available()
    train_ds = FolkMusicCQTDataset("train", use_precomputed)
    test_ds = FolkMusicCQTDataset("test", use_precomputed)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=pin, persistent_workers=num_workers > 0)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=pin, persistent_workers=num_workers > 0)

    print(f"Train samples: {len(train_ds)} | Test samples: {len(test_ds)}")
    return train_loader, test_loader

In [7]:
# =============================================================================
# STEP 1 — PRECOMPUTE FEATURES
# =============================================================================
preprocess_all_dataset_splits(force=False)

# Verify one sample
sample_npy = next(PROCESSED_CQT_DIR.glob("train/*/*.npy"), None)
if sample_npy:
    arr = np.load(sample_npy)
    print(f"Sample feature shape: {arr.shape} (expected ~ (2, {N_BINS}, {TARGET_FRAMES}))")
else:
    print("No precomputed files found yet — dataset will fall back to on-the-fly CQT.")

Starting HPSS + CQT precomputation...

Processing 'train' split...
  tamang_selo: 533 files


    tamang_selo:   0%|          | 0/533 [00:00<?, ?it/s]

  deuda: 528 files


    deuda:   0%|          | 0/528 [00:00<?, ?it/s]

  bhajan: 413 files


...

  tharu: 110 files


    tharu:   0%|          | 0/110 [00:00<?, ?it/s]

  lok_dohori: 124 files


    lok_dohori:   0%|          | 0/124 [00:00<?, ?it/s]


Precomputation done — new: 3713, skipped: 0, errors: 0
Stored in: /kaggle/working/processed_cqt
Sample feature shape: (2, 84, 1292) (expected ~ (2, 84, 1292))


In [8]:
# =============================================================================
# TRAINING — SpecAugment + AdamW + ReduceLROnPlateau
# =============================================================================
class SpecAugment:
    def __init__(self, freq_mask_max=10, time_mask_max=30, num_freq_masks=2, num_time_masks=2):
        self.freq_mask_max = freq_mask_max
        self.time_mask_max = time_mask_max
        self.num_freq_masks = num_freq_masks
        self.num_time_masks = num_time_masks

    def __call__(self, spec):
        spec = spec.clone()
        for _ in range(self.num_freq_masks):
            if spec.shape[1] > self.freq_mask_max:
                f = np.random.randint(0, self.freq_mask_max)
                f0 = np.random.randint(0, spec.shape[1] - f)
                spec[:, f0:f0 + f, :] = 0
        for _ in range(self.num_time_masks):
            if spec.shape[2] > self.time_mask_max:
                t = np.random.randint(0, self.time_mask_max)
                t0 = np.random.randint(0, spec.shape[2] - t)
                spec[:, :, t0:t0 + t] = 0
        return spec


def train_epoch(model, loader, optimizer, criterion, device, spec_augment=None):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for features, labels in tqdm(loader, desc="Train", leave=False):
        features, labels = features.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        if spec_augment:
            features = spec_augment(features)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / len(loader), 100.0 * correct / total


def evaluate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for features, labels in tqdm(loader, desc="Eval", leave=False):
            features, labels = features.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            outputs = model(features)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            preds = outputs.argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=GENRES)
    cm = confusion_matrix(all_labels, all_preds)
    return running_loss / len(loader), acc, report, cm


def save_checkpoint(model, optimizer, epoch, loss, path):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": loss,
        "genres": GENRES,
    }, path)
    print(f"Saved: {path}")


def plot_confusion_matrix(cm, save_path, title="Confusion Matrix"):
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=GENRES, yticklabels=GENRES)
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(title)
    plt.tight_layout(); plt.savefig(save_path, dpi=150); plt.close()


# --- Run training ---
train_loader, test_loader = create_sota_dataloaders()
model = FolkMusicCQTResNet(num_classes=len(GENRES), dropout_prob=0.2).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

sched_kwargs = {"mode": "min", "factor": 0.5, "patience": 5}
if "verbose" in inspect.signature(optim.lr_scheduler.ReduceLROnPlateau).parameters:
    sched_kwargs["verbose"] = True
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, **sched_kwargs)
spec_augment = SpecAugment()

best_path = MODELS_DIR / "sota_best_model.pth"
best_acc = 0.0
history = {"train_loss": [], "test_loss": [], "train_acc": [], "test_acc": []}

print(f"\nTraining {NUM_EPOCHS} epochs on {DEVICE}...")
t0 = time.time()

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print("-" * 50)
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE, spec_augment)
    te_loss, te_acc, report, cm = evaluate_epoch(model, test_loader, criterion, DEVICE)
    scheduler.step(te_loss)

    history["train_loss"].append(tr_loss)
    history["test_loss"].append(te_loss)
    history["train_acc"].append(tr_acc)
    history["test_acc"].append(te_acc * 100)

    print(f"Train Loss: {tr_loss:.4f} | Train Acc: {tr_acc:.2f}%")
    print(f"Test Loss:  {te_loss:.4f} | Test Acc:  {te_acc * 100:.2f}%")
    print(report)

    if te_acc > best_acc:
        best_acc = te_acc
        save_checkpoint(model, optimizer, epoch, te_loss, best_path)
        plot_confusion_matrix(cm, RESULTS_DIR / "sota_confusion_matrix.png")

    if (epoch + 1) % 10 == 0:
        save_checkpoint(model, optimizer, epoch, te_loss, MODELS_DIR / f"sota_checkpoint_epoch_{epoch + 1}.pth")

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="Train"); axes[0].plot(history["test_loss"], label="Test")
axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(history["train_acc"], label="Train"); axes[1].plot(history["test_acc"], label="Test")
axes[1].set_title("Accuracy (%)"); axes[1].legend()
plt.tight_layout(); plt.savefig(RESULTS_DIR / "sota_training_curves.png", dpi=150); plt.close()

print(f"\nTraining complete in {(time.time() - t0) / 60:.1f} min")
print(f"Best test accuracy: {best_acc * 100:.2f}%")

Train samples: 2903 | Test samples: 810

Training 25 epochs on cuda...

Epoch 1/25
--------------------------------------------------


Train:   0%|          | 0/182 [00:00<?, ?it/s]

Eval:   0%|          | 0/51 [00:00<?, ?it/s]

Train Loss: 1.6152 | Train Acc: 35.89%
Test Loss:  1.8627 | Test Acc:  38.64%
              precision    recall  f1-score   support

 tamang_selo       0.50      0.18      0.26       154
       deuda       0.28      0.28      0.28       141
      bhajan       0.63      0.86      0.72       166
      newari       0.45      0.04      0.08       115
       tharu       1.00      0.02      0.04       110
  lok_dohori       0.26      0.78      0.39       124

    accuracy                           0.39       810
   macro avg       0.52      0.36      0.29       810
weighted avg       0.51      0.39      0.32       810

Saved: /kaggle/working/models/sota_best_model.pth

Epoch 2/25
--------------------------------------------------


Train:   0%|          | 0/182 [00:00<?, ?it/s]

...

Eval:   0%|          | 0/51 [00:00<?, ?it/s]

Train Loss: 0.7009 | Train Acc: 88.87%
Test Loss:  1.1702 | Test Acc:  71.85%
              precision    recall  f1-score   support

 tamang_selo       0.70      0.83      0.76       154
       deuda       0.87      0.59      0.70       141
      bhajan       0.82      0.86      0.84       166
      newari       0.55      0.78      0.65       115
       tharu       0.79      0.38      0.52       110
  lok_dohori       0.68      0.78      0.73       124

    accuracy                           0.72       810
   macro avg       0.74      0.70      0.70       810
weighted avg       0.74      0.72      0.71       810


Epoch 25/25
--------------------------------------------------


Train:   0%|          | 0/182 [00:00<?, ?it/s]

Eval:   0%|          | 0/51 [00:00<?, ?it/s]

Train Loss: 0.6985 | Train Acc: 89.29%
Test Loss:  1.4021 | Test Acc:  60.62%
              precision    recall  f1-score   support

 tamang_selo       0.82      0.80      0.81       154
       deuda       0.77      0.40      0.52       141
      bhajan       1.00      0.25      0.40       166
      newari       0.35      0.90      0.51       115
       tharu       0.71      0.57      0.63       110
  lok_dohori       0.65      0.84      0.73       124

    accuracy                           0.61       810
   macro avg       0.72      0.63      0.60       810
weighted avg       0.74      0.61      0.60       810


Training complete in 54.5 min
Best test accuracy: 73.21%


In [9]:
# =============================================================================
# STEP 3 — COMPREHENSIVE EVALUATION
# =============================================================================

def load_model(path, device):
    model = FolkMusicCQTResNet(num_classes=len(GENRES), dropout_prob=0.2).to(device)
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    print(f"Loaded model from {path} (epoch {ckpt.get('epoch', '?')})")
    return model


def full_evaluation(model, loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for features, labels in tqdm(loader, desc="Full eval"):
            features = features.to(device, non_blocking=True)
            outputs = model(features)
            probs = torch.softmax(outputs, dim=1)
            preds = outputs.argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    conf = all_probs.max(axis=1)
    correct = all_preds == all_labels

    metrics = {
        "accuracy": accuracy_score(all_labels, all_preds),
        "precision_macro": precision_score(all_labels, all_preds, average="macro", zero_division=0),
        "recall_macro": recall_score(all_labels, all_preds, average="macro", zero_division=0),
        "f1_macro": f1_score(all_labels, all_preds, average="macro", zero_division=0),
        "precision_per_class": precision_score(all_labels, all_preds, average=None, zero_division=0),
        "recall_per_class": recall_score(all_labels, all_preds, average=None, zero_division=0),
        "f1_per_class": f1_score(all_labels, all_preds, average=None, zero_division=0),
        "confusion_matrix": confusion_matrix(all_labels, all_preds),
        "avg_confidence": conf.mean(),
        "correct_confidence": conf[correct].mean() if correct.any() else 0.0,
        "incorrect_confidence": conf[~correct].mean() if (~correct).any() else 0.0,
        "all_preds": all_preds, "all_labels": all_labels, "all_probs": all_probs,
    }
    return metrics


eval_model = load_model(best_path, DEVICE)
_, test_loader = create_sota_dataloaders()
metrics = full_evaluation(eval_model, test_loader, DEVICE)

print("\n=== EVALUATION SUMMARY ===")
print(f"Accuracy     : {metrics['accuracy']:.4f}")
print(f"Macro F1     : {metrics['f1_macro']:.4f}")
print(f"Avg conf     : {metrics['avg_confidence']:.4f}")
print(f"Correct conf : {metrics['correct_confidence']:.4f}")
print(f"Wrong conf   : {metrics['incorrect_confidence']:.4f}")

plot_confusion_matrix(metrics["confusion_matrix"], RESULTS_DIR / "sota_eval_confusion_matrix.png", "Final Confusion Matrix")

# Per-class bar chart
x = np.arange(len(GENRES)); w = 0.25
plt.figure(figsize=(14, 6))
plt.bar(x - w, metrics["precision_per_class"], w, label="Precision")
plt.bar(x, metrics["recall_per_class"], w, label="Recall")
plt.bar(x + w, metrics["f1_per_class"], w, label="F1")
plt.xticks(x, GENRES, rotation=45, ha="right"); plt.ylim(0, 1.1); plt.legend(); plt.grid(alpha=0.3)
plt.title("Per-Class Metrics"); plt.tight_layout()
plt.savefig(RESULTS_DIR / "sota_per_class_metrics.png", dpi=150); plt.close()

# Confidence distribution
conf = metrics["all_probs"].max(axis=1)
correct = metrics["all_preds"] == metrics["all_labels"]
plt.figure(figsize=(10, 5))
plt.hist(conf[correct], bins=25, alpha=0.7, label="Correct", color="green")
plt.hist(conf[~correct], bins=25, alpha=0.7, label="Incorrect", color="red")
plt.xlabel("Confidence"); plt.ylabel("Count"); plt.legend(); plt.title("Confidence Distribution")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "sota_confidence_distribution.png", dpi=150); plt.close()

report_lines = [
    "# SOTA Evaluation Report\n",
    f"- Accuracy: {metrics['accuracy']:.4f}\n",
    f"- Macro F1: {metrics['f1_macro']:.4f}\n",
    f"- Avg confidence: {metrics['avg_confidence']:.4f}\n",
]
for i, g in enumerate(GENRES):
    report_lines.append(f"- {g}: P={metrics['precision_per_class'][i]:.3f} R={metrics['recall_per_class'][i]:.3f} F1={metrics['f1_per_class'][i]:.3f}\n")
(RESULTS_DIR / "sota_evaluation_report.md").write_text("".join(report_lines))
print(f"Reports saved to {RESULTS_DIR}")

Loaded model from /kaggle/working/models/sota_best_model.pth (epoch 14)
Train samples: 2903 | Test samples: 810


Full eval:   0%|          | 0/51 [00:00<?, ?it/s]


=== EVALUATION SUMMARY ===
Accuracy     : 0.7321
Macro F1     : 0.7179
Avg conf     : 0.7691
Correct conf : 0.8235
Wrong conf   : 0.6204
Reports saved to /kaggle/working/results


In [10]:
# =============================================================================
# STEP 4 — INFERENCE DEMO (chunked prediction on a test file)
# =============================================================================
class SOTAPredictor:
    def __init__(self, model_path, device=DEVICE):
        self.device = device
        self.genre_names = GENRES
        self.model = FolkMusicCQTResNet(num_classes=len(GENRES), dropout_prob=0.2).to(device)
        ckpt = torch.load(model_path, map_location=device, weights_only=False)
        self.model.load_state_dict(ckpt["model_state_dict"])
        self.model.eval()

    def _split_chunks(self, audio, sr, overlap=0.5):
        chunk_len = int(DURATION * sr)
        step = int(chunk_len * (1 - overlap))
        chunks = []
        for start in range(0, len(audio), step):
            chunk = audio[start:start + chunk_len]
            if len(chunk) < chunk_len:
                chunk = np.pad(chunk, (0, chunk_len - len(chunk)))
            chunks.append(chunk)
        return chunks

    def predict_file(self, audio_path, show_chunks=False):
        audio, sr = librosa.load(str(audio_path), sr=SAMPLE_RATE, mono=True)
        chunks = self._split_chunks(audio, sr)
        chunk_preds, chunk_confs = [], []

        for i, chunk in enumerate(chunks):
            feats = pad_or_truncate(compute_sota_features(chunk, sr))
            tensor = torch.tensor(feats, dtype=torch.float32).unsqueeze(0).to(self.device)
            with torch.no_grad():
                probs = torch.softmax(self.model(tensor), dim=1)
                conf, pred = probs.max(1)
            chunk_preds.append(pred.item())
            chunk_confs.append(conf.item())
            if show_chunks:
                print(f"  Chunk {i+1}: {self.genre_names[pred.item()]} ({conf.item():.3f})")

        avg_probs = np.zeros(len(self.genre_names))
        for idx in range(len(self.genre_names)):
            confs = [chunk_confs[i] for i, p in enumerate(chunk_preds) if p == idx]
            avg_probs[idx] = np.mean(confs) if confs else 0.0

        final_idx = int(np.argmax(avg_probs))
        consistency = sum(1 for p in chunk_preds if p == final_idx) / max(len(chunk_preds), 1)
        entropy = -np.sum(avg_probs * np.log(avg_probs + 1e-12))
        reliable = avg_probs[final_idx] >= 0.6 and consistency >= 0.6 and entropy <= 1.0

        return {
            "file": str(audio_path),
            "predicted_genre": self.genre_names[final_idx] if reliable else None,
            "confidence": float(avg_probs[final_idx]),
            "consistency": float(consistency),
            "entropy": float(entropy),
            "is_reliable": reliable,
            "probabilities": {g: float(avg_probs[i]) for i, g in enumerate(self.genre_names)},
            "num_chunks": len(chunks),
        }


predictor = SOTAPredictor(best_path, DEVICE)

# Pick first available test wav
demo_wav = next(LEGACY_SPLITS_DIR.glob("test/*/*.wav"), None)
if demo_wav:
    print(f"Demo prediction on: {demo_wav.name}")
    result = predictor.predict_file(demo_wav, show_chunks=True)
    print("\n=== PREDICTION ===")
    for k, v in result.items():
        if k != "probabilities":
            print(f"{k}: {v}")
    print("Probabilities:")
    for g, p in sorted(result["probabilities"].items(), key=lambda x: -x[1]):
        print(f"  {g}: {p:.3f}")
else:
    print("No test wav found for demo prediction.")

Demo prediction on: saba_nidhaye_prabhu_old_bhaktaraj_bhajan_chunk2.wav
  Chunk 1: bhajan (0.863)
  Chunk 2: bhajan (0.532)

=== PREDICTION ===
file: /kaggle/input/datasets/siddhanta98/cadence-dataset/splits/test/bhajan/saba_nidhaye_prabhu_old_bhaktaraj_bhajan_chunk2.wav
predicted_genre: bhajan
confidence: 0.6974989771842957
consistency: 1.0
entropy: 0.2512769581291835
is_reliable: True
num_chunks: 2
Probabilities:
  bhajan: 0.697
  tamang_selo: 0.000
  deuda: 0.000
  newari: 0.000
  tharu: 0.000
  lok_dohori: 0.000
